# Customer Churn Prediction
## End-to-End Machine Learning Pipeline

**Objective:** Build a predictive model that identifies telecom customers at risk of churning, with explainable predictions to guide retention strategies.

**Pipeline:**
1. Data Loading & Inspection
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training & Comparison
5. Hyperparameter Tuning
6. Model Evaluation
7. Model Explainability (SHAP)
8. Save Model for Deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap
import joblib

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

## 1. Data Loading & Inspection

In [ ]:
# Load the Telco Customer Churn dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Dataset shape: {df.shape}')
print(f'Number of customers: {df.shape[0]:,}')
print(f'Number of features: {df.shape[1]}')
df.head()

In [ ]:
# Data types and missing values
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Dataset Info ===')
df.info()

In [ ]:
# Statistical summary
df.describe(include='all').T

In [ ]:
# TotalCharges has spaces - convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many missing values this created
print(f'Missing TotalCharges after conversion: {df["TotalCharges"].isnull().sum()}')

# These are new customers (tenure=0) - fill with 0
print(f'Tenure of customers with missing TotalCharges:')
print(df[df['TotalCharges'].isnull()]['tenure'].value_counts())
df['TotalCharges'].fillna(0, inplace=True)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, count + 50, f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'], colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 13})
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nChurn rate: {churn_pct["Yes"]:.1f}% — This is an imbalanced dataset.')

In [ ]:
# Churn by contract type — the strongest predictor
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack()

fig, ax = plt.subplots(figsize=(10, 5))
contract_churn.plot(kind='bar', color=colors, edgecolor='black', ax=ax)
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['Retained', 'Churned'], loc='upper right')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=11)

plt.tight_layout()
plt.savefig('../images/churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Churn by tenure — survival analysis style
fig, ax = plt.subplots(figsize=(12, 5))

for label, group_df in df.groupby('Churn'):
    color = '#e74c3c' if label == 'Yes' else '#2ecc71'
    ax.hist(group_df['tenure'], bins=30, alpha=0.6, label=f'Churn={label}',
            color=color, edgecolor='black')

ax.set_title('Customer Tenure Distribution by Churn Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Number of Customers')
ax.legend()
plt.tight_layout()
plt.savefig('../images/tenure_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Customers are most likely to churn in the first few months.')

In [ ]:
# Monthly charges distribution by churn status
fig, ax = plt.subplots(figsize=(12, 5))

df[df['Churn'] == 'No']['MonthlyCharges'].plot(kind='kde', label='Retained',
                                                 color='#2ecc71', linewidth=2, ax=ax)
df[df['Churn'] == 'Yes']['MonthlyCharges'].plot(kind='kde', label='Churned',
                                                  color='#e74c3c', linewidth=2, ax=ax)

ax.set_title('Monthly Charges Distribution by Churn Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/monthly_charges_kde.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Churned customers tend to have higher monthly charges.')

In [ ]:
# Churn rate across all service features
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    axes[i].barh(churn_rate.index, churn_rate.values, color='#3498db', edgecolor='black')
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_xlabel('Churn Rate (%)')
    for j, v in enumerate(churn_rate.values):
        axes[i].text(v + 0.5, j, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig('../images/churn_by_services.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap for numerical features
numerical_df = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
numerical_df['Churn'] = (df['Churn'] == 'Yes').astype(int)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(numerical_df.corr(), annot=True, cmap='RdYlGn_r', center=0,
            fmt='.2f', square=True, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
# Create a copy for modeling
data = df.copy()

# Drop customerID — not useful for prediction
data.drop('customerID', axis=1, inplace=True)

# Encode target variable
data['Churn'] = (data['Churn'] == 'Yes').astype(int)

# Binary columns: convert Yes/No to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    data[col] = (data[col] == 'Yes').astype(int)

# Gender encoding
data['gender'] = (data['gender'] == 'Male').astype(int)

print('Binary encoding complete.')
print(f'Shape: {data.shape}')

In [ ]:
# One-hot encode multi-category columns
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                  'Contract', 'PaymentMethod']

data = pd.get_dummies(data, columns=multi_cat_cols, drop_first=True)

print(f'Shape after one-hot encoding: {data.shape}')
print(f'\nFeatures: {list(data.columns)}')

In [ ]:
# Split features and target
X = data.drop('Churn', axis=1)
y = data['Churn']

# Train-test split (80-20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set: {X_test.shape[0]:,} samples')
print(f'\nChurn rate in training set: {y_train.mean():.3f}')
print(f'Churn rate in test set: {y_test.mean():.3f}')

In [ ]:
# Scale numerical features
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Feature scaling applied to:', num_cols)

In [ ]:
# Handle class imbalance with SMOTE
print(f'Before SMOTE:')
print(f'  Not Churned: {(y_train == 0).sum():,}')
print(f'  Churned: {(y_train == 1).sum():,}')

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE:')
print(f'  Not Churned: {(y_train_balanced == 0).sum():,}')
print(f'  Churned: {(y_train_balanced == 1).sum():,}')

## 4. Model Training & Comparison

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42, use_label_encoder=False, eval_metric='logloss'
    )
}

# Train and evaluate each model
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\n{'='*50}')
    print(f'Training: {name}')
    print('='*50)
    
    # Cross-validation on balanced training data
    cv_scores = cross_val_score(model, X_train_balanced, y_train_balanced,
                                 cv=cv, scoring='roc_auc', n_jobs=-1)
    
    # Fit on full balanced training set
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predict on original (non-SMOTE) test set
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        'test_auc': auc,
        'f1': f1,
        'accuracy': acc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f'CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    print(f'Test ROC-AUC: {auc:.4f}')
    print(f'Test F1-Score: {f1:.4f}')
    print(f'Test Accuracy: {acc:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Retained", "Churned"])}')

In [ ]:
# Model comparison visualization
comparison_df = pd.DataFrame({
    'Model': results.keys(),
    'CV AUC': [r['cv_auc_mean'] for r in results.values()],
    'Test AUC': [r['test_auc'] for r in results.values()],
    'F1 Score': [r['f1'] for r in results.values()],
    'Accuracy': [r['accuracy'] for r in results.values()]
}).set_index('Model')

fig, ax = plt.subplots(figsize=(12, 5))
comparison_df.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0.5, 1.0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(comparison_df.round(4).to_string())

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(10, 7))

colors_roc = ['#3498db', '#2ecc71', '#e74c3c']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['test_auc']:.3f})",
            linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../images/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Tuning (XGBoost)

In [ ]:
# Select best model (XGBoost) and tune hyperparameters
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_tuned = XGBClassifier(
    random_state=42, use_label_encoder=False, eval_metric='logloss'
)

grid_search = GridSearchCV(
    xgb_tuned, param_grid, cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train_balanced, y_train_balanced)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate tuned model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print('=== Tuned XGBoost Performance ===')
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_tuned):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'\n{classification_report(y_test, y_pred_tuned, target_names=["Retained", "Churned"])}')

## 6. Model Evaluation — Detailed

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_tuned)
axes[1].plot(recall, precision, linewidth=2, color='#e74c3c')
axes[1].fill_between(recall, precision, alpha=0.2, color='#e74c3c')
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../images/evaluation_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Explainability (SHAP)

In [ ]:
# SHAP explainer
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values computed for {X_test.shape[0]} test samples.')

In [ ]:
# Global feature importance — SHAP summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False, max_display=15)
plt.title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm plot — shows direction of feature effects
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title('Feature Effects on Churn Prediction (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual prediction explanation — waterfall plot
# Show a high-risk customer
high_risk_idx = y_pred_proba_tuned.argmax()

print(f'Explaining prediction for customer at index {high_risk_idx}')
print(f'Churn probability: {y_pred_proba_tuned[high_risk_idx]:.1%}')
print(f'Actual churn: {"Yes" if y_test.iloc[high_risk_idx] == 1 else "No"}')

plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx],
    feature_names=X_test.columns.tolist()
), max_display=10, show=False)
plt.title('Why This Customer Is Predicted to Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Model for Deployment

In [ ]:
# Save model, scaler, and feature names
joblib.dump(best_model, '../models/churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(list(X.columns), '../models/feature_names.pkl')

print('Model artifacts saved:')
print('  - models/churn_model.pkl')
print('  - models/scaler.pkl')
print('  - models/feature_names.pkl')
print(f'\nModel is ready for deployment via Streamlit app (src/app.py).')

---

## Summary

### Key Results
- **Best Model:** XGBoost (tuned) with ROC-AUC on test set
- **Class imbalance** handled via SMOTE oversampling
- **Top predictors of churn:** Contract type, tenure, monthly charges, tech support, online security

### Business Recommendations
1. **Target month-to-month customers** with incentives to switch to longer contracts
2. **Offer bundled security & tech support** — customers without these services churn at higher rates
3. **Focus retention efforts on the first 12 months** — the highest-risk period
4. **Review pricing strategy** — high monthly charges correlate with churn

### Next Steps
- Deploy the model via Streamlit for real-time predictions
- Monitor model performance over time (concept drift)
- A/B test retention strategies on model-identified high-risk customers